In [2]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import pandas as pd

load_dotenv()  # carga el .env

# Conexión a la BD origen (saleshealth)
engine_origen = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME_ORIGEN')}"
)

# Conexión a la BD destino (DWH) — la usaremos más adelante
engine_dwh = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME_DWH')}"
)

# Test de conexión
with engine_origen.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM sale_item"))
    print(f"✅ Conexión OK — sale_item: {result.scalar()} filas")

✅ Conexión OK — sale_item: 42555 filas


In [3]:
query = """
SELECT 
    table_name,
    (SELECT COUNT(*) FROM information_schema.columns 
     WHERE table_name = t.table_name 
     AND table_schema = 'public') AS num_columnas
FROM information_schema.tables t
WHERE table_schema = 'public' 
AND table_type = 'BASE TABLE'
ORDER BY table_name;
"""

tablas = pd.read_sql(query, engine_origen)

# Añadir conteo de filas por tabla
filas = []
with engine_origen.connect() as conn:
    for tabla in tablas['table_name']:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {tabla}")).scalar()
        filas.append(count)

tablas['num_filas'] = filas
print(tablas.to_string(index=False))

        table_name  num_columnas  num_filas
             brand             4         29
          category             3          6
 central_inventory             8         49
   central_product             8         49
         city_zone             6         42
          customer             7       5750
         inventory             5       1000
             offer             6          1
           product             6         50
     product_offer             2          6
       return_item             5       2330
     return_reason             3          6
              sale             5      20000
         sale_item             7      42555
             store             8         20
         warehouse             7          1
warehouse_location             6         40


In [4]:
tablas_clave = ['sale', 'sale_item', 'customer', 'product', 'store']

for tabla in tablas_clave:
    print(f"\n{'='*50}")
    print(f"TABLA: {tabla}")
    print('='*50)
    df = pd.read_sql(f"SELECT * FROM {tabla} LIMIT 3", engine_origen)
    print(df.to_string())
    print(f"Columnas: {list(df.columns)}")


TABLA: sale
   sale_id  customer_id  store_id                  sale_date   total
0        3          160        15 2023-09-23 10:23:26.643082  359.98
1        4          298         5 2020-10-18 02:06:58.749665  399.99
2        5          212        20 2021-02-03 18:50:47.573929  399.84
Columnas: ['sale_id', 'customer_id', 'store_id', 'sale_date', 'total']

TABLA: sale_item
   sale_item_id  sale_id  product_id  quantity  unit_price offer_id  subtotal
0             1        3          46         2      179.99     None    359.98
1             2        4          45         1      399.99     None    399.99
2             3        5          35         1       59.99     None     59.99
Columnas: ['sale_item_id', 'sale_id', 'product_id', 'quantity', 'unit_price', 'offer_id', 'subtotal']

TABLA: customer
   customer_id first_name last_name last_name2                         email          phone                 created_at
0            1      Marta      Ruiz     García       marta.ruiz1@example

In [5]:
query_fk = """
SELECT
    tc.table_name AS tabla_origen,
    kcu.column_name AS columna_fk,
    ccu.table_name AS tabla_destino,
    ccu.column_name AS columna_pk
FROM information_schema.table_constraints tc
JOIN information_schema.key_column_usage kcu
    ON tc.constraint_name = kcu.constraint_name
JOIN information_schema.constraint_column_usage ccu
    ON tc.constraint_name = ccu.constraint_name
WHERE tc.constraint_type = 'FOREIGN KEY'
AND tc.table_schema = 'public'
ORDER BY tc.table_name;
"""

fks = pd.read_sql(query_fk, engine_origen)
print(fks.to_string(index=False))

      tabla_origen   columna_fk      tabla_destino   columna_pk
 central_inventory   product_id    central_product   product_id
 central_inventory  location_id warehouse_location  location_id
 central_inventory warehouse_id          warehouse warehouse_id
   central_product     brand_id              brand     brand_id
   central_product  category_id           category  category_id
         inventory   product_id            product   product_id
         inventory     store_id              store     store_id
     product_offer   product_id            product   product_id
     product_offer     offer_id              offer     offer_id
       return_item sale_item_id          sale_item sale_item_id
              sale  customer_id           customer  customer_id
              sale     store_id              store     store_id
         sale_item   product_id            product   product_id
         sale_item     offer_id              offer     offer_id
         sale_item      sale_id         

In [6]:
print(pd.read_sql("SELECT * FROM city_zone LIMIT 3", engine_origen))

  postal_code   district   area_type zone_orientation city_code    city
0       28030  Moratalaz  Periférica             None        28  Madrid
1       28001  Salamanca    Céntrica           Centro        28  Madrid
2       28004     Centro    Céntrica           Centro        28  Madrid


In [7]:
for tabla in ['product', 'central_product', 'brand', 'category', 
              'store', 'offer', 'product_offer', 'return_reason']:
    df = pd.read_sql(f"SELECT * FROM {tabla} LIMIT 2", engine_origen)
    print(f"\n{tabla}: {list(df.columns)}")


product: ['product_id', 'name', 'category', 'manufacturer', 'price', 'created_at']

central_product: ['product_id', 'name', 'category_id', 'brand_id', 'sku', 'barcode', 'unit_cost', 'unit_price']

brand: ['brand_id', 'name', 'country', 'website']

category: ['category_id', 'name', 'description']

store: ['store_id', 'name', 'address', 'city', 'postal_code', 'latitude', 'longitude', 'opened_date']

offer: ['offer_id', 'name', 'description', 'discount_percent', 'start_date', 'end_date']

product_offer: ['product_id', 'offer_id']

return_reason: ['reason_id', 'reason', 'active']


In [8]:
print(pd.read_sql("SELECT * FROM customer LIMIT 3", engine_origen))

   customer_id first_name last_name last_name2                         email  \
0            1      Marta      Ruiz     García       marta.ruiz1@example.com   
1            2  Alejandro     López      López  alejandro.lópez2@example.com   
2            3  Alejandro     López    Álvarez  alejandro.lópez3@example.com   

           phone                 created_at  
0    +34 6987791 2018-11-15 04:51:29.131242  
1  +34 669926134 2018-09-07 20:45:28.751233  
2  +34 661558537 2019-02-09 06:54:03.656551  


In [9]:
print(pd.read_sql("SELECT * FROM return_item LIMIT 3", engine_origen))

   return_id  sale_item_id                return_date  quantity  reason_id
0       1935         40347 2025-11-07 08:32:07.541770         1          5
1       1936         40496 2025-10-31 00:52:24.854172         1          5
2       1937         40093 2024-08-19 05:54:58.092921         1          1


In [10]:
print(pd.read_sql("SELECT * FROM customer LIMIT 2", engine_origen).columns.tolist())
print(pd.read_sql("SELECT * FROM city_zone LIMIT 2", engine_origen).columns.tolist())

['customer_id', 'first_name', 'last_name', 'last_name2', 'email', 'phone', 'created_at']
['postal_code', 'district', 'area_type', 'zone_orientation', 'city_code', 'city']


In [11]:
print(pd.read_sql("SELECT * FROM sale LIMIT 3", engine_origen).columns.tolist())
print(pd.read_sql("SELECT * FROM store LIMIT 3", engine_origen))

['sale_id', 'customer_id', 'store_id', 'sale_date', 'total']
   store_id             name            address    city postal_code  latitude  \
0         1     Store Retiro    Calle Alcalá 85  Madrid       28009   40.4214   
1         2  Store Salamanca      Calle Goya 45  Madrid       28001   40.4252   
2         3  Store Chamartín  Calle Serrano 210  Madrid       28016   40.4582   

   longitude opened_date  
0    -3.6740  2026-04-04  
1    -3.6837  2026-04-04  
2    -3.6840  2026-04-04  


In [12]:
print(pd.read_sql("SELECT * FROM sale_item LIMIT 2", engine_origen).columns.tolist())
print(pd.read_sql("SELECT * FROM central_product LIMIT 2", engine_origen).columns.tolist())

['sale_item_id', 'sale_id', 'product_id', 'quantity', 'unit_price', 'offer_id', 'subtotal']
['product_id', 'name', 'category_id', 'brand_id', 'sku', 'barcode', 'unit_cost', 'unit_price']


In [13]:
df_si = pd.read_sql("SELECT DISTINCT product_id FROM sale_item", engine_origen)
df_cp = pd.read_sql("SELECT DISTINCT product_id FROM central_product", engine_origen)

faltantes = set(df_si['product_id']) - set(df_cp['product_id'])
print(f"Products en sale_item pero NO en central_product: {faltantes}")

# Ver cuántas ventas tienen esos productos
df_check = pd.read_sql("SELECT product_id, COUNT(*) as n FROM sale_item GROUP BY product_id ORDER BY product_id", engine_origen)
print(df_check[df_check['product_id'].isin(faltantes)])

Products en sale_item pero NO en central_product: {29}
    product_id    n
28          29  711


In [14]:
print(pd.read_sql("SELECT * FROM product WHERE product_id = 29", engine_origen))

   product_id                            name        category manufacturer  \
0          29  Sensor temperatura inteligente  Domótica Salud       Xiaomi   

   price                 created_at  
0  19.99 2026-04-04 11:23:48.591695  
